In [ ]:
import scanpy as sc
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import scipy

from matplotlib.colors import LinearSegmentedColormap
import matplotlib.cm as cm

from sccoda.util import comp_ana as mod
from sccoda.util import cell_composition_data as dat
from sccoda.util import data_visualization as viz

# from anndata import AnnData
import anndata as ad

from misc_utils import score_colors

In [ ]:
# data_annotated_path = "/home/bmb/haxx/working/ceisel_mumm/data/"
data_annotated_path = "/Users/bbrener1/haxx/ceisel_mumm/data/"
data = sc.read_h5ad(data_annotated_path + "full_annotations_leiden_old.h5ad")
data = data[data.obs["cell_type"] != "trash"]
data.obs['time'] = data.obs['time'].astype(str)

In [ ]:
cell_counts = (
    data.obs
    .groupby(["exp_condition", "time", "cell_type",])
    .size()
    .unstack(fill_value=0)  
)

covariate_df = cell_counts.index.to_frame(index=False)

sccoda_data = ad.AnnData(
    X=cell_counts.values,
    obs=covariate_df.reset_index(drop=True),
    var=pd.DataFrame(index=cell_counts.columns.astype(str))
)

In [ ]:
sccoda_data.obs['batch'] = sccoda_data.obs['time'].apply(lambda x: "0" if x == "12" or x == "72" else "1") # If we add the batchmark before, the counts table will have gaps

In [ ]:
sccoda_data.obs

In [ ]:
# model = mod.CompositionalAnalysis(sccoda_data, formula="exp_condition + time")
# model = mod.CompositionalAnalysis(sccoda_data, formula="exp_condition + time + batch")
# model = mod.CompositionalAnalysis(sccoda_data, formula="exp_condition + time + exp_condition:time")

In [ ]:
sim_results = model.sample_hmc()

In [ ]:
with pd.option_context('display.max_rows', None, 'display.max_columns', None):
    print(sim_results.summary())
    print(sim_results.credible_effects())